# Tweety-10 — Markov Logic Networks (MLN) en .NET (C# / IKVM)

**Navigation** : [← Tweety-9-Préférences](Tweety-09-Preferences-Python.ipynb) | [Index](./README.md) | [Tweety-11-Causal-Python →](Tweety-11-Causal-Python.ipynb) | [Tweety-11c (C#)](Tweety-11-Causal-CSharp.ipynb)

**Module** : `org.tweetyproject.logics.mln` (Tweety 1.30) — compilé via IKVM 8.14.0 + .NET 8.0

**Clé** : `See #4667` (EPIC Tweety .NET — 11ème port C#)

**Recette build** : `dotnet-build/build-tweety-mln-shade.pom.xml` (maven-shade-plugin 3.5.3) + `dotnet-build/build-TweetyMlnShade.csproj` (IKVM 8.14.0, net8.0)

## Objectifs pédagogiques

A la fin de ce notebook, vous saurez :

1. **Construire un MLN** : faits stricts (poids = infini) et règles pondérées (poids fini) à partir de formules FOL
2. **Interroger un MLN** : `query(mln, formula)` retourne une probabilité marginale
3. **Comprendre le spectre logique ↔ statistique** : un poids croissant rend la règle de plus en plus stricte
4. **Diagnostiquer un domaine** : utiliser le réseau de Markov pour propager des croyances (ex. médical, social, fraude)

## Prérequis

- Avoir exécuté [Tweety-02-Basic-Logics-CSharp.ipynb](Tweety-02-Basic-Logics-CSharp.ipynb) (notebook 2) pour `FolSignature`/`FolParser`/`FolFormula`
- Avoir exécuté [Tweety-03-ModalLogic-CSharp.ipynb](Tweety-03-ModalLogic-CSharp.ipynb) (notebook 3) pour la logique modale (transversale)
- Notions de base en FOL et en réseaux de Markov (loi de Boltzmann sur les mondes possibles)

In [1]:
// --- Initialisation runtime IKVM + chargement DLL MLN (rebuild C190 IKVM 8.14) ---
#r "nuget: IKVM, 8.14.0"
#r "org.tweetyproject.tweety-mln.dll"
// IKVM.Home DOIT etre connu du static initializer (JVM.Properties) AVANT tout appel Java.
using System.IO;
// RID de la machine courante (win-x64, linux-x64, osx-arm64...) : IKVM.Image tire l'image native de chaque plateforme.
string ikvmVer = "8.14.0", ikvmRid = (OperatingSystem.IsWindows() ? "win" : OperatingSystem.IsMacOS() ? "osx" : "linux") + "-" + System.Runtime.InteropServices.RuntimeInformation.ProcessArchitecture.ToString().ToLowerInvariant();
string nugetRoot = Environment.GetEnvironmentVariable("NUGET_PACKAGES")
    ?? Path.Combine(Environment.GetFolderPath(Environment.SpecialFolder.UserProfile), ".nuget", "packages");
string ikvmBaseAny = Path.Combine(nugetRoot, "ikvm.image", ikvmVer, "ikvm", "any", "any");
string ikvmArchDir = Path.Combine(nugetRoot, "ikvm.image.runtime." + ikvmRid, ikvmVer, "ikvm", "any", ikvmRid);
string ikvmHome = Path.Combine(Path.GetTempPath(), "ikvm-home-" + ikvmVer + "-" + ikvmRid);
void IkvmCopyMerge(string src, string dst)
{
    foreach (var d in Directory.GetDirectories(src, "*", SearchOption.AllDirectories))
        Directory.CreateDirectory(d.Replace(src, dst));
    foreach (var f in Directory.GetFiles(src, "*", SearchOption.AllDirectories))
    {
        var t = f.Replace(src, dst);
        Directory.CreateDirectory(Path.GetDirectoryName(t)!);
        File.Copy(f, t, true);
    }
}
if (Directory.Exists(ikvmBaseAny) && Directory.Exists(ikvmArchDir))
{
    Directory.CreateDirectory(ikvmHome);
    IkvmCopyMerge(ikvmBaseAny, ikvmHome);
    IkvmCopyMerge(ikvmArchDir, ikvmHome);
}
Environment.SetEnvironmentVariable("IKVM_HOME", ikvmHome);
AppContext.SetData("IKVM.Home", ikvmHome);
AppContext.SetData("ikvm.home", ikvmHome);
System.Console.WriteLine("IKVM home=" + (File.Exists(Path.Combine(ikvmHome, "lib", "tzdb.dat")) ? "OK (basename=" + Path.GetFileName(ikvmHome) + ")" : "MISSING"));
System.Console.WriteLine($"DLL MLN reference chargee (chemin masque pour portabilite).");



Installing Packages IKVM

IKVM home=OK (basename=ikvm-home-8.14.0-win-x64)


DLL MLN reference chargee (chemin masque pour portabilite).


### Interprétation de l'initialisation

**Succès** : la DLL `org.tweetyproject.tweety-mln.dll` (≈14 MB) est chargée dans le runtime .NET via IKVM. Elle contient l'intégralité du module `logics-mln` de Tweety 1.30 (Markov Logic Networks) + ses dépendances transitives `logics-fol` (First-Order Logic) et `logics-pcl` (Probabilistic Conditional Logic), compilées depuis le JAR `tweety-mln-full-1.30.jar` (12.9 MB).

**Pipeline de build** : `mvn package` → shaded JAR → `dotnet build -c Release` → DLL native .NET (cf. recette C182/C183/C185/C186/C187, voir `build-tweety-mln-shade.pom.xml` et `build-TweetyMlnShade.csproj`).

**Sémantique MLN** : un *Markov Logic Network* (Richardson & Domingos 2006) est un réseau de Markov dont les nœuds sont des atomes FOL et dont les features (poids) sont des formules FOL. Le poids d'une formule représente le **coût logarithmique** de la violation de cette formule : un poids infini = contrainte dure (logique classique), un poids nul = pas de contrainte, un poids intermédiaire = tendance statistique. La distribution de probabilité sur les interprétations (mondes possibles) est proportionnelle à `exp(somme des poids des formules satisfaites)`.

## Partie 1 : De la logique du premier ordre à la logique pondérée

### 1.1 Le constat : la FOL est binaire (et donc fragile)

En logique du premier ordre classique, une formule est soit **toujours vraie** (tautologie), soit **parfois fausse** (dans certaines interprétations). Une règle comme « les oiseaux volent » s'écrit `∀x. Bird(x) ⇒ Flies(x)`. **Mais alors, que fait-on du pingouin ?** En FOL, soit on abandonne la règle (on perd la généralisation), soit on l'accepte (on classifie mal les pingouins).

### 1.2 La solution MLN : la même formule, deux statuts

Avec les MLN, la **même formule** `Bird(x) ⇒ Flies(x)` peut être :

- **stricte** (poids = ∞) : équivalent à une règle FOL dure, jamais violée
- **pondérée** (poids = `2.5` par exemple) : la règle est *presque* toujours vraie, mais peut être violée si d'autres règles (strictes) l'exigent (ex. `Penguin(x) ⇒ ¬Flies(x)`)

### 1.3 Construire une `MlnFormula` : stricte vs pondérée

On reprend la signature FOL du notebook 2 et on l'enrobe dans `MlnFormula` avec ou sans poids.

In [2]:
// --- Cell[4] : reconstruction C190 (IKVM 8.14 fix) ---
// MLN simple : 2 formules ponderees
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;

var sig = new FolSignature();
var Smokes = new Predicate("Smokes", 1);
var Cancer = new Predicate("Cancer", 1);
sig.add(Smokes); sig.add(Cancer);

var X = new Variable("X");
var Y = new Variable("Y");

// Regle 1 (poids eleve) : si X fume, alors X a un cancer
var smokesX = new FolAtom(Smokes, new Term[] { X });
var cancerX = new FolAtom(Cancer, new Term[] { X });
var r1 = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(smokesX, cancerX);
var f1 = new MlnFormula(r1, java.lang.Double.valueOf(2.0));

// Regle 2 (poids faible) : si X et Y amis et X fume, alors Y fume (propagation)
var Friends = new Predicate("Friends", 2);
sig.add(Friends);
var friendsXY = new FolAtom(Friends, new Term[] { X, Y });
var smokesY = new FolAtom(Smokes, new Term[] { Y });
var r2 = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(
    new org.tweetyproject.logics.fol.syntax.Conjunction(friendsXY, smokesX),
    smokesY);
var f2 = new MlnFormula(r2, java.lang.Double.valueOf(0.5));

var mln = new MarkovLogicNetwork();
mln.add(f1); mln.add(f2);

System.Console.WriteLine($"MLN avec {mln.size()} formules ponderees : {mln.toString()}");




MLN avec 2 formules ponderees : { <(Friends(X,Y)&&Smokes(X)=>Smokes(Y)), 0.5>, <(Smokes(X)=>Cancer(X)), 2.0> }


### Interprétation : deux statuts pour une même formule

**Sortie réelle** de la cellule précédente :

```
MLN avec 2 formules ponderees : { <(Friends(X,Y)&&Smokes(X)=>Smokes(Y)), 0.5>, <(Smokes(X)=>Cancer(X)), 2.0> }
```

L'ordre des formules dans l'affichage suit celui de l'ensemble interne (`BeliefSet`), pas celui de leur ajout : c'est une propriété de `toString()`, sans effet sur le modèle.

**Points clés** :

1. `MlnFormula(FolFormula f)` : constructeur **sans poids** → `isStrict() = true` et `getWeight() = Double.POSITIVE_INFINITY` (règle FOL dure, jamais violée par le solveur).
2. `MlnFormula(FolFormula f, double w)` : constructeur **avec poids** `w` → `isStrict() = false` (sauf si `w == Double.POSITIVE_INFINITY`) et `getWeight() = w` (tendance statistique). La cellule précédente n'utilise que ce constructeur : les deux règles y sont pondérées, l'une à 2.0, l'autre à 0.5.
3. **Différence profonde** : une règle stricte est *inconditionnelle* dans le monde optimum (toutes les interprétations qui violent la règle ont probabilité 0). Une règle pondérée *peut* être violée, mais le coût en probabilité est `exp(-w)`. Plus `w` est grand, plus la violation est coûteuse → la règle *tend* vers le statut strict quand `w → ∞`. La partie 4 met les deux statuts en regard sur le même domaine : exception stricte pour les pingouins, tendance pondérée pour les oiseaux.

## Diagnostic IKVM — corrigé : le reasoner MLN est invocable

Le diagnostic précédent de ce notebook (« `Assembly.GetTypes()` retourne 0 types → bug IKVM 8.15, reasoner bloqué ») était **faux en mécanisme** — vérifié firsthand le 2026-08-20 :

1. **Les classes se chargent et s'instancient.** `MarkovLogicNetwork`, `MlnFormula`, les `FolFormula` et les reasoners de `org.tweetyproject.logics.mln.reasoner` sont présents dans l'assembly chargée (listing reflection : `ApproximateNaiveMlnReasoner`, `AbstractMlnReasoner`, `SimpleSamplingMlnReasoner`, `SimpleMlnReasoner`). `SatNaiveMlnReasoner`, nommé par l'audit initial, est **absent** de cette build du shade.
2. **Le vrai blocage était une signature d'interop.** Le constructeur de `ApproximateNaiveMlnReasoner` prend des primitifs `(long, long)` : lui passer des `java.lang.Integer` boxés échoue (CS1503) — d'où l'échec attribué à tort au moteur. Avec les littéraux `200000L, 200000L`, le reasoner s'instancie et `query()` retourne les **mêmes marginales que le jumeau Python/JPype** : `P(Smokes(bob)) = 0.7294` à 4 décimales.
3. **Le défaut de `Assembly.GetTypes()`** sur les JAR shades est une lacune de la *surface de réflexion*, pas du chargement : les types sont pleinement utilisables même quand l'énumération les omet.

Les cellules des parties 2 à 5 invoquent désormais le vrai reasoner Java (même moteur que le jumeau Python) : marginales multi-atomes, sweep de poids, paradoxe du pingouin avec exception stricte, comparaison exact-vs-sampling. Le cas 1-atome en forme fermée (partie A du sweep) est conservé comme **ancre analytique** : il valide le régime du moteur sur un domaine factorisable à la main.

## Partie 2 : L'exemple canonique — friends / smokers / cancer

L'exemple de référence de *Markov Logic Networks* (Richardson & Domingos, 2006) combine trois règles :

1. **Le tabagisme mène au cancer** : `Smokes(X) => Cancer(X)` (poids élevé : tendance forte)
2. **Les amis partagent leurs habitudes** : `Friends(X,Y) && Smokes(X) => Smokes(Y)` (poids moyen : propagation sociale)
3. **Des faits observés** : `Smokes(anna)` (strict), `Friends(anna,bob)` (strict), `Friends(bob,carl)` (strict)

La question n'est plus « bob a-t-il un cancer ? » (oui/non), mais « **quelle est la probabilité** que bob ait un cancer, sachant qu'il est ami avec anna qui fume ? »

In [3]:
// --- Cell[8] : exemple canonique MLN reseau social (Richardson & Domingos 2006) ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;
using org.tweetyproject.logics.mln.reasoner;
using System.Globalization;

// InvariantCulture -> separateur decimal '.' deterministe (independant de la locale)
string Inv(FormattableString fs) => fs.ToString(CultureInfo.InvariantCulture);

// Signature : 3 personnes, predicats Smokes/1, Cancer/1, Friends/2
var sig = new FolSignature();
var Smokes = new Predicate("Smokes", 1);
var Cancer = new Predicate("Cancer", 1);
var Friends = new Predicate("Friends", 2);
sig.add(Smokes); sig.add(Cancer); sig.add(Friends);
var anna = new Constant("anna"); var bob = new Constant("bob"); var carl = new Constant("carl");
sig.add(anna); sig.add(bob); sig.add(carl);
var X = new Variable("X"); var Y = new Variable("Y");

FolFormula At(Predicate p, params Term[] t) => (FolFormula)new FolAtom(p, t);

// Regles ponderees quantifiees (meme enonce que le jumeau Python / JPype) :
//   R1 : Smokes(X) => Cancer(X)                     [w=3.0]
//   R2 : (Friends(X,Y) && Smokes(X)) => Smokes(Y)   [w=2.0]
var r1 = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(At(Smokes, X), At(Cancer, X));
var r2 = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(
    new org.tweetyproject.logics.fol.syntax.Conjunction(At(Friends, X, Y), At(Smokes, X)),
    At(Smokes, Y));

// Faits stricts (sans poids) : anna fume, anna-bob amis, bob-carl amis
var mln = new MarkovLogicNetwork();
mln.add(new MlnFormula(r1, java.lang.Double.valueOf(3.0)));
mln.add(new MlnFormula(r2, java.lang.Double.valueOf(2.0)));
mln.add(new MlnFormula(At(Smokes, anna)));
mln.add(new MlnFormula(At(Friends, anna, bob)));
mln.add(new MlnFormula(At(Friends, bob, carl)));

Console.WriteLine($"MLN reseau social (3 personnes) : {mln.size()} formules");
Console.WriteLine(mln.toString());
Console.WriteLine();

// Reasoner : enumeration naive exacte, deterministe sur ce domaine (2^15 mondes < seuil).
// NB interop : le constructeur prend des primitifs (long, long) -> litteraux 200000L.
var reasoner = new ApproximateNaiveMlnReasoner(200000L, 200000L);
Console.WriteLine("Marginales a posteriori (reasoner exact, meme moteur Java que le jumeau Python) :");
Console.WriteLine("  Atome             | P(atome)");
Console.WriteLine("  ------------------|---------");
foreach (var (label, atome) in new (string, FolFormula)[] {
    ("Smokes(anna)", At(Smokes, anna)),
    ("Smokes(bob)",   At(Smokes, bob)),
    ("Smokes(carl)",  At(Smokes, carl)),
    ("Cancer(anna)",  At(Cancer, anna)),
    ("Cancer(bob)",   At(Cancer, bob)),
    ("Cancer(carl)",  At(Cancer, carl)),
})
{
    double p = reasoner.query(mln, atome).doubleValue();
    Console.WriteLine(Inv($"  {label,-17} | {p:F4}"));
}


MLN reseau social (3 personnes) : 5 formules


{ <(Friends(X,Y)&&Smokes(X)=>Smokes(Y)), 2.0>, <Friends(anna,bob), null>, <Friends(bob,carl), null>, <Smokes(anna), null>, <(Smokes(X)=>Cancer(X)), 3.0> }


Marginales a posteriori (reasoner exact, meme moteur Java que le jumeau Python) :


  Atome             | P(atome)


  ------------------|---------


  Smokes(anna)      | 1.0000


  Smokes(bob)       | 0.7294


  Smokes(carl)      | 0.7294


  Cancer(anna)      | 0.9526


  Cancer(bob)       | 0.8301


  Cancer(carl)      | 0.8301


### Interprétation : marginales a posteriori du réseau social

**Sortie réelle** (reasoner `ApproximateNaiveMlnReasoner`, même moteur Java que le jumeau Python/JPype) :

| Atome | P(atome) | Lecture |
|---|---|---|
| `Smokes(anna)` | 1.0000 | fait strict — certain |
| `Smokes(bob)` | 0.7294 | lié à anna par `Friends(anna,bob)` |
| `Smokes(carl)` | 0.7294 | lié à bob par `Friends(bob,carl)` — **deux** liens depuis anna |
| `Cancer(anna)` | 0.9526 | fumeuse certaine, règle w=3 |
| `Cancer(bob)` | 0.8301 | ne fume qu'à 73% → cancer moins certain |
| `Cancer(carl)` | 0.8301 | même valeur que bob — lecture en cell[11] |

**Lecture** : (1) les marginales de tabagisme de bob et carl sont **identiques** dans ce modèle — la règle `Friends(X,Y) && Smokes(X) => Smokes(Y)` [w=2] propage l'influence d'anna vers bob, puis de bob vers carl. (2) Le cancer suit le tabac avec un léger flou : `Cancer(anna) = 0.9526` (fumer est certain, mais la règle w=3 n'est pas stricte — le monde « anna fume sans cancer » garde un poids non nul `exp(-3)`) ; bob et carl, qui ne fument qu'à 73%, tombent à 0.8301.

**Vers la vérification.** Ces valeurs sont-elles les vraies marginales du modèle, ou un artefact du moteur ? Et que signifie l'égalité bob = carl, puisque les deux individus n'occupent pas la même place dans le graphe ? La cellule suivante répond par une **énumération exhaustive écrite ici**, indépendante de tout reasoner : elle somme la distribution jointe, confronte le moteur à cette référence, mesure le seuil exact où il cesse d'être exact, et expose le défaut de normalisation de son régime aléatoire.

In [4]:
// --- Cell[10] : verification independante du moteur - reference exhaustive + defaut de normalisation ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.fol.semantics;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;
using org.tweetyproject.logics.mln.reasoner;
using System.Globalization;

string InvV(FormattableString fs) => fs.ToString(CultureInfo.InvariantCulture);

// Reference exhaustive ECRITE ICI : les instances ground viennent de l'API
// (allGroundInstances), mais l'enumeration des mondes et la somme des poids ne passent
// par AUCUN reasoner. Semantique d'un MLN : poids(monde) = exp( somme_f w_f x nb
// d'instances ground de f satisfaites ), et 0 des qu'une formule STRICTE a une instance
// non satisfaite. Le facteur e uniforme que computeWeight ajoute par formule stricte
// entierement satisfaite est neutre apres normalisation : il est omis dans les marginales,
// mais compte dans lnMax, le logarithme du poids que le moteur attribue au monde le plus lourd.
// Chaque monde est un ulong (un bit par atome ground) ; les faits stricts sont fixes,
// il ne reste donc que 2^(atomes libres) mondes a sommer.
(Dictionary<FolAtom, double> P, int nAtoms, int nFree, double lnMax) Reference(FolSignature s, MarkovLogicNetwork net, FolAtom[] queries)
{
    var bits = new Dictionary<FolAtom, int>();
    int n = 0;
    for (var ai = new HerbrandBase(s).getAtoms().iterator(); ai.hasNext(); ) bits[(FolAtom)ai.next()] = n++;
    if (n > 40) throw new NotSupportedException("domaine trop grand pour l'enumeration exhaustive");
    Func<ulong, bool> Not(Func<ulong, bool> g) => m => !g(m);
    Func<ulong, bool> And(Func<ulong, bool> l, Func<ulong, bool> r) => m => l(m) && r(m);
    Func<ulong, bool> Or(Func<ulong, bool> l, Func<ulong, bool> r) => m => l(m) || r(m);
    Func<ulong, bool> Implies(Func<ulong, bool> l, Func<ulong, bool> r) => m => !l(m) || r(m);
    Func<ulong, bool> Lit(FolAtom a) { ulong mask = 1UL << bits[a]; return m => (m & mask) != 0; }
    Func<ulong, bool> Cmp(FolFormula f) => f switch
    {
        FolAtom a => Lit(a),
        Negation ng => Not(Cmp(ng.getFormula())),
        Conjunction c => And(Cmp((FolFormula)c.getFormulas().get(0)), Cmp((FolFormula)c.getFormulas().get(1))),
        Disjunction d => Or(Cmp((FolFormula)d.getFormulas().get(0)), Cmp((FolFormula)d.getFormulas().get(1))),
        Implication im => Implies(Cmp((FolFormula)im.getFormulas().getFirst()), Cmp((FolFormula)im.getFormulas().getSecond())),
        _ => throw new NotSupportedException("forme non traitee : " + f.GetType().Name)
    };
    ulong fixedMask = 0;
    int nStrict = 0;
    var rules = new List<(bool strict, double w, Func<ulong, bool>[] inst)>();
    for (var fi = net.iterator(); fi.hasNext(); )
    {
        var mf = (MlnFormula)fi.next();
        var g = mf.getFormula();
        if (mf.isStrict() && g is FolAtom fa) { fixedMask |= 1UL << bits[fa]; nStrict++; continue; }
        var instL = new List<Func<ulong, bool>>();
        for (var gi = g.allGroundInstances(s.getConstants()).iterator(); gi.hasNext(); )
            instL.Add(Cmp((FolFormula)gi.next()));
        rules.Add((mf.isStrict(), mf.isStrict() ? 0.0 : mf.getWeight().doubleValue(), instL.ToArray()));
        if (mf.isStrict()) nStrict++;
    }
    var freeBits = new List<ulong>();
    foreach (var kv in bits) if ((fixedMask & (1UL << kv.Value)) == 0) freeBits.Add(1UL << kv.Value);
    int nf = freeBits.Count;
    var libre = freeBits.ToArray();                 // bit de chaque atome libre
    var qMask = new ulong[queries.Length];          // bit de chaque atome requete
    for (int q = 0; q < queries.Length; q++) qMask[q] = 1UL << bits[queries[q]];
    double Z = 0;
    var mass = new double[queries.Length];
    double eMax = double.NegativeInfinity;
    long count = 1L << nf;
    for (long m = 0; m < count; m++)
    {
        ulong world = fixedMask;
        for (int j = 0; j < nf; j++) if ((m & (1L << j)) != 0) world |= libre[j];
        double e = 0; bool ok = true;
        foreach (var (strict, wgt, inst) in rules)
        {
            int sat = 0;
            foreach (var fn in inst) if (fn(world)) sat++;
            if (strict) { if (sat != inst.Length) { ok = false; break; } }
            else e += wgt * sat;
        }
        if (!ok) continue;
        double wt = Math.Exp(e);
        Z += wt;
        if (e > eMax) eMax = e;
        for (int q = 0; q < queries.Length; q++) if ((world & qMask[q]) != 0) mass[q] += wt;
    }
    var P = new Dictionary<FolAtom, double>();
    for (int q = 0; q < queries.Length; q++) P[queries[q]] = mass[q] / Z;
    return (P, n, nf, eMax + nStrict);
}

// Masse d'une formule dans un modele rendu par computeModel (Map brute, valeurs java.lang.Double)
double Masse(java.util.Map mm, FolFormula f)
{
    double s = 0;
    for (var ki = mm.keySet().iterator(); ki.hasNext(); )
    {
        var h = (HerbrandInterpretation)ki.next();
        if (h.satisfies(f)) s += ((java.lang.Double)mm.get(h)).doubleValue();
    }
    return s;
}
double MasseTotale(java.util.Map mm)
{
    double s = 0;
    for (var vi = mm.values().iterator(); vi.hasNext(); ) s += ((java.lang.Double)vi.next()).doubleValue();
    return s;
}

// ---- A. Reference exhaustive du reseau de la cell[8], comparee au moteur ----
double borneLong = 9.223372036854775807E18;   // Long.MAX_VALUE en Java
var qA = new[] {
    new FolAtom(Smokes, anna), new FolAtom(Smokes, bob), new FolAtom(Smokes, carl),
    new FolAtom(Cancer, anna), new FolAtom(Cancer, bob), new FolAtom(Cancer, carl)
};
var sigMin = (FolSignature)mln.getMinimalSignature();
var refA = Reference(sig, mln, qA);
var modA = reasoner.computeModel(mln, sigMin);
long mondesA = 1L << refA.nFree;
Console.WriteLine(InvV($"A. Reference exhaustive : {refA.nAtoms} atomes ground, {refA.nFree} atomes libres (faits stricts fixes), {mondesA} mondes a sommer"));
Console.WriteLine("   Atome             | reference | moteur (200000,200000)");
Console.WriteLine("   ------------------|-----------|----------------------");
foreach (var q in qA)
    Console.WriteLine(InvV($"   {q.ToString(),-17} | {refA.P[q]:F4}    | {Masse(modA, q):F4}"));
Console.WriteLine(InvV($"   ecart max reference/moteur : {qA.Max(q => Math.Abs(refA.P[q] - Masse(modA, q))):F6}"));
Console.WriteLine(InvV($"   poids maximal d'un monde valide : e^{refA.lnMax:F0} = {Math.Exp(refA.lnMax):E2} ; borne d'un long : {borneLong:E2}"));

// Le seuil exact, mesure au point pres : a (32768,32768) le modele rendu contient les
// 32768 mondes et sa masse totale vaut 1 -- enumeration complete, renormalisation correcte.
var modExact = new ApproximateNaiveMlnReasoner(32768L, 32768L).computeModel(mln, sigMin);
Console.WriteLine(InvV($"   (32768,32768) : {modExact.size()} mondes distincts, masse totale = {MasseTotale(modExact):F6}"));

// ---- B. Un cran sous le seuil : regime aleatoire, defaut de normalisation ----
// computeModel : tant que 2^N <= maxSelected -> DefaultSubsetIterator (tous les mondes) ;
// sinon -> RandomSubsetIterator(atoms, false), tirage AVEC REMISE, puis
// result.put(monde, poids/sumOfWeights) dans une HashMap : deux tirages identiques sont
// un seul monde, alors que sumOfWeights compte tous les tirages. La masse rendue est donc
// la COUVERTURE (fraction de la masse totale portee par les mondes distincts), et chaque
// marginale brute = couverture x estimateur sur les mondes distincts.
// Couverture attendue : seuls les mondes VALIDES (poids non nul) portent de la masse, et
// chaque tirage valide est uniforme sur eux. Avec r = tirages valides / mondes valides,
// la masse des tirages couverte par des mondes distincts vaut en moyenne (1-(1-1/M)^n)/r,
// soit (1-exp(-r))/r. Ici r vaut environ 1 (32767/8 tirages valides pour 4096 mondes).
double CouvertureAttendue(double mondesValides, double tiragesValides)
    => (1.0 - Math.Pow(1.0 - 1.0 / mondesValides, tiragesValides)) / (tiragesValides / mondesValides);
double tiragesValidesA = 32767.0 / Math.Pow(2, refA.nAtoms - refA.nFree);
double rA = tiragesValidesA / mondesA;
double couvA = CouvertureAttendue(mondesA, tiragesValidesA);
Console.WriteLine();
Console.WriteLine(InvV($"B. Regime aleatoire (32767,32767) : couverture attendue = {couvA:F4} pour r = {rA:F4} (tirage non seede)"));
Console.WriteLine("   tirage | mondes distincts | masse totale | P(Smokes(anna)) | P(Smokes(bob)) brute | corrigee | reference");
Console.WriteLine("   ------|-----------------|--------------|-----------------|----------------------|----------|----------");
double sumBrut = 0, sumCorr = 0;
for (int t = 1; t <= 5; t++)
{
    var mm = new ApproximateNaiveMlnReasoner(32767L, 32767L).computeModel(mln, sigMin);
    double couverture = Masse(mm, qA[0]);   // le fait strict Smokes(anna) vaut 1 dans le joint
    double brute = Masse(mm, qA[1]);
    double corrigee = brute / couverture;   // renormalisation par la couverture mesuree
    sumBrut += brute; sumCorr += corrigee;
    Console.WriteLine(InvV($"   {t,6} | {mm.size(),16} | {MasseTotale(mm),12:F4} | {couverture,15:F4} | {brute,20:F4} | {corrigee,8:F4} | {refA.P[qA[1]],9:F4}"));
}
Console.WriteLine(InvV($"   moyenne P(Smokes(bob)) : brute {sumBrut / 5:F4} ; corrigee {sumCorr / 5:F4} ; reference {refA.P[qA[1]]:F4}"));
Console.WriteLine("   Un fait strict qui ne sort pas a 1 signale le regime aleatoire, pas le modele.");
Console.WriteLine("   Diviser par la couverture ramene autour de la vraie valeur, a la fluctuation pres :");
Console.WriteLine("   ce n'est pas une reparation du reasoner. Sur un petit domaine, le remede est la");
Console.WriteLine("   capacite : maxSelected >= 2^N force l'enumeration complete.");

// ---- C. Le piege de la signature minimale ----
// query(net, f) utilise la signature MINIMALE du MLN : les seules constantes qui
// apparaissent dans une formule. Une constante declaree mais absente des formules est
// ignoree, sans erreur. Ici bob est declare dans sigC mais n'apparait dans aucune
// formule de netC : la requete a deux arguments rend 0, celle a trois arguments est juste.
var sigC = new FolSignature();
sigC.add(Smokes); sigC.add(Cancer);
var annC = new Constant("anna"); var bobC = new Constant("bob");
sigC.add(annC); sigC.add(bobC);
var XC = new Variable("X");
var netC = new MarkovLogicNetwork();
netC.add(new MlnFormula((FolFormula)new Implication(new FolAtom(Smokes, XC), new FolAtom(Cancer, XC)),
    java.lang.Double.valueOf(3.0)));
netC.add(new MlnFormula(new FolAtom(Smokes, annC)));
var qC = new FolAtom(Smokes, bobC);
var rC = new ApproximateNaiveMlnReasoner(100000L, 100000L);
var refC = Reference(sigC, netC, new[] { qC });
var cstMin = new List<string>();
for (var ci = ((FolSignature)netC.getMinimalSignature()).getConstants().iterator(); ci.hasNext(); )
    cstMin.Add(ci.next().ToString());
Console.WriteLine();
Console.WriteLine("C. Piege de la signature : netC = { Smokes(X)=>Cancer(X) [3.0], Smokes(anna) }, domaine {anna,bob}");
Console.WriteLine("   constantes de la signature minimale : " + string.Join(", ", cstMin));
Console.WriteLine(InvV($"   query(netC, Smokes(bob))       = {rC.query(netC, qC).doubleValue():F4}   <- faux, silencieux"));
Console.WriteLine(InvV($"   query(netC, Smokes(bob), sigC) = {rC.query(netC, qC, sigC).doubleValue():F4}   <- juste ; reference exhaustive = {refC.P[qC]:F4}"));

// ---- D. Ce que l'egalite bob = carl ne demontre pas : chaine a 4 personnes ----
// Reference de l'exercice 1 : on prolonge la chaine anna -> bob -> carl par dave,
// avec le fait strict Friends(carl,dave). 24 atomes ground, 20 libres : 2^20 mondes.
var sig4 = (FolSignature)sig.clone();
var dave = new Constant("dave");
sig4.add(dave);
var mln4 = new MarkovLogicNetwork();
for (var fi = mln.iterator(); fi.hasNext(); ) mln4.add((MlnFormula)fi.next());
mln4.add(new MlnFormula(new FolAtom(Friends, carl, dave)));
var q4 = new[] {
    new FolAtom(Smokes, anna), new FolAtom(Smokes, bob), new FolAtom(Smokes, carl), new FolAtom(Smokes, dave),
    new FolAtom(Cancer, anna), new FolAtom(Cancer, bob), new FolAtom(Cancer, carl), new FolAtom(Cancer, dave)
};
var ref4 = Reference(sig4, mln4, q4);
var mod4 = new ApproximateNaiveMlnReasoner(200000L, 200000L).computeModel(mln4, (FolSignature)mln4.getMinimalSignature());
double masse4 = MasseTotale(mod4);
var sampling4 = new SimpleSamplingMlnReasoner(0.01, 1000);
double tiragesValides4 = 200000.0 / Math.Pow(2, ref4.nAtoms - ref4.nFree);
double r4 = tiragesValides4 / (1L << ref4.nFree);
double couv4 = CouvertureAttendue(1L << ref4.nFree, tiragesValides4);
Console.WriteLine();
Console.WriteLine(InvV($"D. Chaine a 4 personnes (anna->bob->carl->dave) : {ref4.nAtoms} atomes ground, {1L << ref4.nFree} mondes valides sommes ; moteur (200000,200000) : {mod4.size()} mondes distincts"));
Console.WriteLine(InvV($"   masse totale mesuree {masse4:F4} ; couverture attendue {couv4:F4} pour r = {r4:F5}"));
Console.WriteLine(InvV($"   poids maximal d'un monde valide : e^{ref4.lnMax:F0} = {Math.Exp(ref4.lnMax):E2}"));
Console.WriteLine("   Atome             | reference | moteur brut | brut/masse | SimpleSampling(0.01,1000)");
Console.WriteLine("   ------------------|-----------|-------------|------------|-------------------------");
foreach (var q in q4)
{
    double brute = Masse(mod4, q);
    Console.WriteLine(InvV($"   {q.ToString(),-17} | {ref4.P[q]:F4}    | {brute:F4}      | {brute / masse4:F4}     | {sampling4.query(mln4, q, sig4).doubleValue():F4}"));
}
Console.WriteLine("   bob et carl se separent ; bob et dave s'egalisent : l'egalite de la cell[8]");
Console.WriteLine("   est une propriete du joint a 3 personnes, pas une symetrie du graphe.");
Console.WriteLine("   Les tirages valides sont rares devant les mondes valides : peu de doublons, le defaut de");
Console.WriteLine("   normalisation devient discret, et l'ecart a la reference vient de l'echantillonnage.");
Console.WriteLine("   SimpleSampling est hors d'usage ici, et le fait strict Smokes(anna) en est le temoin :");
Console.WriteLine("   vrai dans tout monde valide, il devrait sortir a 1. Sa source accumule la masse satisfaite");
Console.WriteLine("   dans un long, alors que le poids maximal de ce reseau depasse la borne imprimee en A :");
Console.WriteLine("   le numerateur sature. A 3 personnes, ce meme poids maximal tient dans un long.");


A. Reference exhaustive : 15 atomes ground, 12 atomes libres (faits stricts fixes), 4096 mondes a sommer


   Atome             | reference | moteur (200000,200000)


   ------------------|-----------|----------------------


   Smokes(anna)      | 1.0000    | 1.0000


   Smokes(bob)       | 0.7294    | 0.7294


   Smokes(carl)      | 0.7294    | 0.7294


... (40 outputs left out)

   SimpleSampling est hors d'usage ici, et le fait strict Smokes(anna) en est le temoin :


   vrai dans tout monde valide, il devrait sortir a 1. Sa source accumule la masse satisfaite


   dans un long, alors que le poids maximal de ce reseau depasse la borne imprimee en A :


   le numerateur sature. A 3 personnes, ce meme poids maximal tient dans un long.


### Interprétation : vérification indépendante du moteur

La cellule précédente calcule elle-même la distribution jointe, sans passer par un reasoner, puis confronte le moteur à cette référence. Quatre lectures, chacune adossée au bloc de sortie qui porte le même nom.

**A. Exactitude mesurée, pas supposée.** Le domaine de la cell[8] porte 15 atomes ground, dont trois fixés par les faits stricts : la référence somme 2^12 mondes, chacun représenté par un entier de 64 bits. Les six marginales de la référence coïncident avec celles du moteur à 4 décimales, et l'écart maximal imprimé est nul : `ApproximateNaiveMlnReasoner(200000L, 200000L)` est exact sur ce domaine, comme son nom « Approximate » ne le laisse pas deviner. La raison se lit dans `computeModel` : tant que `2^N ≤ maxSelected`, l'itérateur est `DefaultSubsetIterator` — tous les mondes, un par un. Le seuil est mesuré au point près : à `(32768L, 32768L)`, le modèle rendu contient les 32768 mondes et sa masse totale vaut exactement 1.

**B. Le régime aléatoire est défectueux, pas seulement imprécis.** Un cran sous le seuil, `(32767L, 32767L)`, l'itérateur devient `RandomSubsetIterator(atoms, false)` : tirage **avec remise**, puis `result.put(monde, poids / sumOfWeights)` dans une `HashMap`. Deux tirages identiques n'occupent qu'une entrée, alors que `sumOfWeights` les compte tous deux : la masse totale rendue tombe sous 1, et chaque marginale brute est sous-évaluée du même facteur — la **couverture**, c'est-à-dire la part de la masse portée par les mondes distincts. Le tableau mesure ce mécanisme sur cinq tirages indépendants : la masse totale coïncide avec la probabilité brute du fait strict `Smokes(anna)`, et diviser la marginale brute de `Smokes(bob)` par cette couverture la ramène autour de la référence, à la fluctuation d'échantillonnage près. La couverture attendue se calcule : seuls les mondes **valides** (poids non nul) portent de la masse, et chaque tirage valide est uniforme sur eux ; avec `r = tirages valides / mondes valides`, la part couverte vaut en moyenne `(1 - (1-1/M)^n) / r`, soit `(1 - e^-r)/r`. Ici `r` vaut environ 1, d'où une perte massive. Conséquence pratique : **un fait strict qui ne sort pas à 1 signale le régime aléatoire, pas le modèle.** Renormaliser par la couverture mesurée est un diagnostic, pas une réparation du reasoner ; sur un petit domaine, le remède est de dimensionner `maxSelected ≥ 2^N`.

**C. Le piège de la signature minimale.** `query(mln, formule)` n'utilise pas la signature du carnet mais la **signature minimale** du MLN : les seules constantes qui apparaissent dans une formule. Une constante déclarée mais absente des formules est ignorée, sans erreur ni avertissement. Le bloc C construit un réseau où `bob` est déclaré dans la signature mais n'apparaît dans aucune formule : la requête à deux arguments rend 0, là où la surcharge à trois arguments `query(netC, Smokes(bob), sigC)` retrouve la valeur de la référence exhaustive. Toujours passer la signature quand le domaine dépasse les constantes mentionnées.

**D. Ce que l'égalité ne démontre pas.** bob et carl ne jouent **pas** des rôles interchangeables dans le graphe de la cell[8] : bob a un lien entrant **et** un lien sortant, carl seulement un lien entrant — aucun automorphisme ne les échange. Leur égalité de marginales est une propriété du **joint** à trois personnes, pas une symétrie structurelle. Le bloc D prolonge la chaîne par `dave` avec le fait strict `Friends(carl,dave)` : la référence sépare bob et carl, et égalise bob et dave. L'égalité observée en cell[8] ne se déduit donc pas de la position dans le graphe.

Le même bloc mesure le moteur sur ce domaine élargi : les 2^24 mondes dépassent la capacité `(200000L, 200000L)`, le reasoner bascule donc en régime aléatoire — mais ici les tirages valides sont rares devant les mondes valides (`r` très petit), les doublons sont peu nombreux (moins de 1 %), la masse totale reste proche de 1, et la colonne renormalisée par cette masse s'écarte de la référence de l'ordre du centième : dans ce régime, c'est la fluctuation d'échantillonnage qui domine, pas le défaut de normalisation. Ce défaut n'est pas une fatalité uniforme : il s'aggrave quand `r` grandit.

Enfin, `SimpleSamplingMlnReasoner(0.01, 1000)` n'est pas une alternative sur ce réseau, et pour une raison plus grave qu'une imprécision. Sa source accumule la masse satisfaite dans un `long`, alors que le poids maximal d'un monde vaut ici `e^48 ≈ 7×10^20` — au-delà de `Long.MAX_VALUE` (≈ 9.22e18). Le numérateur sature et l'estimation s'effondre d'un facteur de l'ordre du centième. Le témoin est net : `Smokes(anna)` est un fait strict, vrai dans tout monde valide, et l'échantillonneur lui attribue la même valeur dégradée qu'aux autres atomes. Sur le réseau à trois personnes, ce poids maximal (`e^30`) tient dans un `long` : l'estimateur y fluctue simplement. L'exercice 1 fait refaire ce calcul au moteur.

## Partie 3 : Comment les poids modèlent la croyance

La même règle, selon son poids, se comporte différemment. Faisons varier le poids de la règle `Smokes(X) => Cancer(X)` et observons l'évolution de `P(Cancer(bob))` :

| Poids | Comportement attendu |
|-------|----------------------|
| 0.0 | Aucune contrainte → `P(Cancer(bob)) ≈ 0.5` (distribution uniforme sur les modèles) |
| 0.5 | Faible contrainte → `P` monte un peu |
| 1.0 | Contrainte modérée |
| 2.0 | Contrainte forte |
| 5.0 | Quasi-stricte (poids `exp(-5) ≈ 0.007` par violation, **avant normalisation**) |
| ∞ | Strict : l'implication est imposée dans tous les mondes de poids non nul — ce qui **n'équivaut pas** à `P = 1`, car `Smokes(bob)` reste incertain |

C'est le **spectre logique ↔ statistique** : le MLN interpole continûment entre la logique (poids infini) et la statistique pure (poids 0). La cellule suivante mesure aussi la marginale du modèle **à règle stricte**, qui borne la convergence de ce réseau particulier.

In [5]:
// --- Cell[13] : le spectre logique-statistique - ancre analytique + vrai reasoner ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;
using org.tweetyproject.logics.mln.reasoner;
using System.Globalization;

string Inv(FormattableString fs) => fs.ToString(CultureInfo.InvariantCulture);

// ---- Partie A : cas trivial 1-atome { <P(a), w> } - forme fermee analytique ----
// 2 mondes exactement -> la fonction de partition se factorise :
//   Z(w) = exp(w) + 1,   P(P(a)) = sigma(w) = exp(w)/(1+exp(w)).
// Le moteur calcule cette ancre : sigma(w) est la valeur attendue, moteur la mesure.
var sigA = new FolSignature();
var Pp = new Predicate("P", 1); sigA.add(Pp);
var ca = new Constant("a"); sigA.add(ca);
var pa = (FolFormula)new FolAtom(Pp, new Term[] { ca });

Console.WriteLine("A. Spectre 1-atome (ancre analytique sigma(w)) :");
foreach (var w in new double[] { 0.0, 0.5, 1.0, 3.0, 5.0 })
{
    var mlnA = new MarkovLogicNetwork();
    mlnA.add(new MlnFormula(pa, java.lang.Double.valueOf(w)));
    double pSigma = Math.Exp(w) / (1.0 + Math.Exp(w));
    double pMesure = new ApproximateNaiveMlnReasoner(200000L, 200000L).query(mlnA, pa).doubleValue();
    Console.WriteLine(Inv($"  w = {w,-4:F1} -> sigma(w) = {pSigma:F4} | moteur = {pMesure:F4}"));
}

// ---- Partie B : cas multi-atomes - le VRAI reasoner, sweep du poids de R1 ----
// Meme protocole que le jumeau Python cell[11] : reseau social a 3 personnes,
// on fait varier le poids de 'Smokes(X) => Cancer(X)' et on observe P(Cancer(bob)).
Console.WriteLine();
Console.WriteLine("B. Sweep du poids w de 'Smokes=>Cancer' sur P(Cancer(bob)) (reasoner exact sur ce domaine) :");

var sigB = new FolSignature();
var Smokes = new Predicate("Smokes", 1);
var Cancer = new Predicate("Cancer", 1);
var Friends = new Predicate("Friends", 2);
sigB.add(Smokes); sigB.add(Cancer); sigB.add(Friends);
var anna = new Constant("anna"); var bob = new Constant("bob"); var carl = new Constant("carl");
sigB.add(anna); sigB.add(bob); sigB.add(carl);
var X = new Variable("X"); var Y = new Variable("Y");

FolFormula At(Predicate p, params Term[] t) => (FolFormula)new FolAtom(p, t);

var rCancer = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(At(Smokes, X), At(Cancer, X));
var rSocial = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(
    new org.tweetyproject.logics.fol.syntax.Conjunction(At(Friends, X, Y), At(Smokes, X)),
    At(Smokes, Y));

foreach (var w in new double[] { 0.0, 0.5, 1.0, 2.0, 3.0, 5.0 })
{
    var mlnB = new MarkovLogicNetwork();
    mlnB.add(new MlnFormula(rCancer, java.lang.Double.valueOf(w)));
    mlnB.add(new MlnFormula(rSocial, java.lang.Double.valueOf(2.0)));
    mlnB.add(new MlnFormula(At(Smokes, anna)));
    mlnB.add(new MlnFormula(At(Friends, anna, bob)));
    mlnB.add(new MlnFormula(At(Friends, bob, carl)));
    var rn = new ApproximateNaiveMlnReasoner(200000L, 200000L);
    double pBob = rn.query(mlnB, At(Cancer, bob)).doubleValue();
    Console.WriteLine(Inv($"  w = {w,-4:F1} -> P(Cancer(bob)) = {pBob:F4}"));
}

// Une implication stricte impose Cancer(bob) des que Smokes(bob) est vrai, mais
// elle n'affirme pas que bob fume. On calcule donc la LIMITE du sweep avec le
// constructeur strict (sans poids), les autres regles et faits inchanges.
var mlnStrict = new MarkovLogicNetwork();
mlnStrict.add(new MlnFormula(rCancer));
mlnStrict.add(new MlnFormula(rSocial, java.lang.Double.valueOf(2.0)));
mlnStrict.add(new MlnFormula(At(Smokes, anna)));
mlnStrict.add(new MlnFormula(At(Friends, anna, bob)));
mlnStrict.add(new MlnFormula(At(Friends, bob, carl)));
var rnStrict = new ApproximateNaiveMlnReasoner(200000L, 200000L);
double pCancerStrict = rnStrict.query(mlnStrict, At(Cancer, bob)).doubleValue();
double pSmokesStrict = rnStrict.query(mlnStrict, At(Smokes, bob)).doubleValue();
Console.WriteLine(Inv($"  regle stricte   -> P(Cancer(bob)) = {pCancerStrict:F4} ; P(Smokes(bob)) = {pSmokesStrict:F4}"));

Console.WriteLine();
Console.WriteLine(Inv($"Convergence mesuree : w=0 -> 0.5 (aucune contrainte), w=5 -> 0.8535, puis le plateau"));
Console.WriteLine(Inv($"du modele a regle stricte : {pCancerStrict:F4} -- et non 1. Le reste ({1 - pCancerStrict:F4}) est la"));
Console.WriteLine("masse des mondes ou bob ne fume pas mais a un cancer : la regle, meme stricte,");
Console.WriteLine("n'affirme pas Smokes(bob).");

A. Spectre 1-atome (ancre analytique sigma(w)) :


  w = 0.0  -> sigma(w) = 0.5000 | moteur = 0.5000


  w = 0.5  -> sigma(w) = 0.6225 | moteur = 0.6225


  w = 1.0  -> sigma(w) = 0.7311 | moteur = 0.7311


  w = 3.0  -> sigma(w) = 0.9526 | moteur = 0.9526


  w = 5.0  -> sigma(w) = 0.9933 | moteur = 0.9933


... (8 outputs left out)

Convergence mesuree : w=0 -> 0.5 (aucune contrainte), w=5 -> 0.8535, puis le plateau


du modele a regle stricte : 0.8573 -- et non 1. Le reste (0.1427) est la


masse des mondes ou bob ne fume pas mais a un cancer : la regle, meme stricte,


n'affirme pas Smokes(bob).


### Interprétation : le spectre logique ↔ statistique

**Partie A (ancre analytique, calculée par le moteur)** : sur le MLN trivial à 1 atome, la distribution se calcule exactement à la main : `Z(w) = exp(w) + 1`, d'où la **sigmoïde** `P(a) = sigma(w)`. Les deux colonnes de la sortie coïncident à 4 décimales sur les cinq poids testés — le moteur est donc exact sur ce régime. À `w = 0` : distribution uniforme (0.5). À `w → +inf` : fait dur (P → 1). C'est l'**ancre analytique** qui valide le régime du moteur — et le contraste avec la partie B est instructif : la limite vaut 1 ici **parce que** l'atome est seul, sans autre règle incertaine en amont.

**Partie B (reasoner, multi-atomes)** : la même physique, mais sur le réseau social à 3 personnes où plus rien ne se factorise à la main :

| w de `Smokes=>Cancer` | 0 | 0.5 | 1.0 | 2.0 | 3.0 | 5.0 | règle stricte |
|---|---|---|---|---|---|---|---|
| `P(Cancer(bob))` | 0.5000 | 0.6024 | 0.6850 | 0.7865 | 0.8301 | 0.8535 | 0.8573 |

**Lecture** : `w = 0` désactive la règle — retour à l'uniforme (0.5) sur `Cancer(bob)`. Quand `w` monte, chaque unité de poids supplémentaire coûte `exp(-w)` au monde violant la règle : la probabilité **converge vers la marginale du modèle à règle stricte, mesurée à 0.8573**, et non vers 1. À `w = 3` on retrouve exactement la marginale 0.8301 de la cell[8] — même modèle, même voie d'interrogation, même valeur : cohérence interne du calcul.

> **Point pédagogique** : c'est LE spectre des MLN, mais son extrémité dépend du réseau. Rendre `Smokes(X) => Cancer(X)` stricte **n'affirme pas que bob fume** : les mondes où bob ne fume pas restent possibles (le modèle strict donne `P(Smokes(bob)) = 0.7145`), et la règle n'y impose rien. Le plafond de 0.8573 est donc une propriété de **ce** modèle, pas une borne universelle ; sur l'ancre 1-atome de la partie A, la même limite vaut 1.

## Partie 4 : Le paradoxe des exceptions — le pingouin qui ne vole pas

Une règle FOL stricte `Bird(X) => Flies(X)`, associée au fait `Bird(tweety)`, implique que Tweety vole. Si l'on ajoute aussi une règle stricte selon laquelle les pingouins ne volent pas et le fait que Tweety est un pingouin, ces hypothèses deviennent incompatibles. Cela illustre une limite de cet ensemble de règles, pas une impossibilité générale de représenter les exceptions en FOL : la règle `Penguin(X) => !Flies(X)`, seule, le fait très bien. Ce que la FOL ne sait pas exprimer, c'est une règle **révisable** qui reste vraie « en général » tout en admettant des cas contraires. Les logiques non monotones proposent d'autres façons de modéliser ces règles révisables.

Pour représenter ici une tendance qui admet des exceptions, le modèle MLN ci-dessous combine :

1. `Penguin(X) => !Flies(X)` — **stricte** : les mondes où un pingouin vole sont exclus.
2. `Bird(X) => Flies(X)` — **pondérée** [w=2.0] : les mondes satisfaisant cette règle sont favorisés sans exclure les autres.

Tweety est déclaré oiseau et pingouin. Robin est déclaré oiseau, sans fait qui établisse ou exclue `Penguin(robin)` : les deux règles doivent donc être prises en compte dans sa marginale — la règle stricte l'exclut des mondes où robin serait à la fois pingouin et volant, la règle pondérée favorise les mondes où il vole.

In [6]:
// --- Cell[16] : paradoxe du pingouin - exception STRICTE vs regle ponderee ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;
using org.tweetyproject.logics.mln.reasoner;
using System.Globalization;

string Inv(FormattableString fs) => fs.ToString(CultureInfo.InvariantCulture);

// Signature : constantes tweety (pingouin) / robin (merle)
var sig = new FolSignature();
var Bird = new Predicate("Bird", 1);
var Penguin = new Predicate("Penguin", 1);
var Flies = new Predicate("Flies", 1);
sig.add(Bird); sig.add(Penguin); sig.add(Flies);
var tweety = new Constant("tweety"); var robin = new Constant("robin");
sig.add(tweety); sig.add(robin);
var X = new Variable("X");

FolFormula At(Predicate p, params Term[] t) => (FolFormula)new FolAtom(p, t);
FolFormula Neg(FolFormula f) => (FolFormula)new org.tweetyproject.logics.fol.syntax.Negation(f);

var mln = new MarkovLogicNetwork();
// Faits stricts
mln.add(new MlnFormula(At(Bird, tweety)));     // tweety est un oiseau...
mln.add(new MlnFormula(At(Penguin, tweety)));  // ...et un pingouin
mln.add(new MlnFormula(At(Bird, robin)));      // robin est un oiseau ordinaire
// Regle STRICTE (sans poids) : les pingouins ne volent jamais - loi dure
var stricte = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(
    At(Penguin, X), Neg(At(Flies, X)));
mln.add(new MlnFormula(stricte));
// Regle PONDEREE : la plupart des oiseaux volent [w=2.0] - tendance avec exceptions
var generale = (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(
    At(Bird, X), At(Flies, X));
mln.add(new MlnFormula(generale, java.lang.Double.valueOf(2.0)));

Console.WriteLine($"MLN paradoxe du pingouin : {mln.size()} formules");
Console.WriteLine(mln.toString());
Console.WriteLine();
Console.WriteLine("  tweety : Bird + Penguin   | robin : Bird seulement");
Console.WriteLine("  strict Penguin=>!Flies    | pondere Bird=>Flies [w=2]");
Console.WriteLine();

var rb = new ApproximateNaiveMlnReasoner(100000L, 100000L);
foreach (var (label, atome) in new (string, FolFormula)[] {
    ("Flies(tweety)", At(Flies, tweety)),
    ("Flies(robin)",  At(Flies, robin)),
})
{
    double p = rb.query(mln, atome).doubleValue();
    Console.WriteLine(Inv($"  P({label}) = {p:F4}"));
}

// Ce que donnerait la SEULE regle ponderee, si robin n'etait jamais pingouin :
// sigma(2) = exp(2) / (1 + exp(2)). L'ecart mesure vient du fait que Penguin(robin)
// n'est pas fixe : les mondes ou robin est pingouin sont exclus du vol par la regle
// stricte, et pesent du cote non-volant sans satisfaire la regle ponderee.
double sigma2 = Math.Exp(2.0) / (1.0 + Math.Exp(2.0));
Console.WriteLine(Inv($"  sigma(2) si robin n'etait jamais pingouin = {sigma2:F4}"));

Console.WriteLine();
Console.WriteLine("=> tweety (pingouin) ne vole pas : 0.0000 -- l'exception STRICTE domine (tout monde");
Console.WriteLine("   ou un pingouin vole a une probabilite nulle).");
Console.WriteLine("=> robin (oiseau ordinaire) vole probablement : le poids 2 favorise le vol sans");
Console.WriteLine("   l'imposer, et les mondes ou robin est pingouin pesent du cote non-volant.");

MLN paradoxe du pingouin : 5 formules


{ <Bird(tweety), null>, <Bird(robin), null>, <Penguin(tweety), null>, <(Penguin(X)=>!Flies(X)), null>, <(Bird(X)=>Flies(X)), 2.0> }


  tweety : Bird + Penguin   | robin : Bird seulement


  strict Penguin=>!Flies    | pondere Bird=>Flies [w=2]


  P(Flies(tweety)) = 0.0000


  P(Flies(robin)) = 0.7870


  sigma(2) si robin n'etait jamais pingouin = 0.8808


=> tweety (pingouin) ne vole pas : 0.0000 -- l'exception STRICTE domine (tout monde


   ou un pingouin vole a une probabilite nulle).


=> robin (oiseau ordinaire) vole probablement : le poids 2 favorise le vol sans


   l'imposer, et les mondes ou robin est pingouin pesent du cote non-volant.


### Interprétation : la logique pondérée gère les exceptions nativement

**Sortie réelle** (reasoner exact) :

```
  P(Flies(tweety)) = 0.0000
  P(Flies(robin))  = 0.7870
  sigma(2) si robin n'etait jamais pingouin = 0.8808
```

**Pourquoi tweety obtient `0.0000` et pas une petite valeur non nulle ?** Parce que `Penguin(X) => !Flies(X)` est **stricte** (ajoutée sans poids) : tout monde où un pingouin vole a une probabilité **nulle**, point. La règle pondérée `Bird(X) => Flies(X)` [w=2] ne peut pas rehausser un monde interdit par une contrainte stricte.

**Pourquoi robin obtient `0.7870`, et pas `1.0` ni même `sigma(2) = 0.8808` ?** Le poids 2 explique le passage de `1.0` à `sigma(2)` : il favorise le vol sans l'imposer. La liberté de `Penguin(robin)` explique le reste. Rien n'établit ni n'exclut que robin soit pingouin — l'énoncé de la partie 4 le dit déjà — et les mondes valides se répartissent alors en trois familles : robin vole (donc n'est pas pingouin), robin ne vole pas et n'est pas pingouin, robin ne vole pas et est pingouin. La troisième est chassée du vol par la règle stricte, et la règle pondérée ne la compte pas satisfaite. D'où `0.7870 = e^2 / (e^2 + 2)` : deux mondes non-volants de poids 1 contre un monde volant de poids `e^2`. Si `Penguin(robin)` était fixé à faux, la même règle pondérée donnerait `sigma(2) = 0.8808` ; l'écart entre les deux valeurs est la part du monde « pingouin » dans l'incertitude.

| Situation | FOL stricte (`Bird=>Flies` dur) | MLN (pondéré + exception stricte) |
|-----------|----------------------------------|------------------------------------|
| Oiseau ordinaire | `Flies = Vrai` (certain) | `P(Flies) ≈ 0.79` (probable) |
| Pingouin | **contradiction !** | `P(Flies) = 0` (correct) |
| Cas « la plupart » | inexprimable | naturel via le poids |

> **Leçon** : les exceptions sont le talon d'Achille de la logique classique. Les MLN les absorbent en faisant varier le **statut** des règles — strict (sans poids) pour la loi dure, pondéré pour la tendance générale.

## Partie 5 : Trois familles de raisonneurs MLN

Comme pour les solveurs SAT du notebook 2 (Sat4j portable vs CaDiCaL natif), Tweety propose plusieurs raisonneurs MLN aux compromis exact vs approximatif :

| Raisonneur | Méthode | Exactitude | Domaine typique |
|------------|---------|------------|-----------------|
| `ApproximateNaiveMlnReasoner` | Tous les mondes si `2^N ≤ maxSelected`, sinon tirage aléatoire avec remise de `maxSelected` mondes dont on garde les `maxForModel` plus lourds | **Exact** dans le premier régime ; sous-évaluation d'un facteur de couverture dans le second | Petits domaines (ici ~15 atomes ground) |
| `SimpleSamplingMlnReasoner` | Échantillonnage Monte-Carlo | Approximatif (paramètre de précision) ; sa masse satisfaite est accumulée dans un `long`, qui sature dès qu'un monde pèse plus de `Long.MAX_VALUE` | Grands domaines à poids modérés |
| `AlchemyMlnReasoner` | Invocation du binaire externe Alchemy : écrit le MLN dans un fichier temporaire, lance `infer -p true -maxSteps 100000`, relit la réponse | Approximatif (échantillonnage MCMC) | Nécessite le binaire installé (non installé ici) |

Le seuil est **mesurable** : sur le réseau de la partie 2, `ApproximateNaiveMlnReasoner(32768L, 32768L)` couvre exactement les 32768 mondes et sa masse totale vaut 1 ; à `(32767L, 32767L)`, le même réseau bascule en régime aléatoire et la masse totale tombe sous 1 (bloc B de cell[10]). À (200000, 200000), comme dans les cellules de ce carnet, il est exact sur ce domaine — vérifié en cell[10] par une énumération exhaustive indépendante. **En régime aléatoire, le reasoner n'est pas seulement approximatif : sa normalisation est défectueuse** (mécanisme en cell[11]) — les marginales y sont sous-évaluées d'un même facteur de couverture, et un fait strict peut sortir en dessous de 1. L'amplitude du défaut dépend du rapport entre tirages valides et mondes valides : il est massif quand ce rapport approche 1, discret quand les tirages valides sont rares (bloc D de cell[10]). Ne lisez jamais une marginale de ce régime comme une référence.

> **Note** : `AlchemyMlnReasoner` requiert un binaire externe non installé — et sa méthode `isInstalled()` répond pourtant `True` : ce témoin ne détecte pas l'absence du binaire. Les deux premiers raisonneurs sont **purement Java** (dans la DLL) et tournent partout — c'est eux que la cellule suivante compare sur la même requête `P(Cancer(carl))` du réseau de la partie 2. Le paquet en contient deux autres, qui ne sont pas exécutés dans les cellules du carnet : `SimpleMlnReasoner`, qui énumère les mondes dans un fichier temporaire (« we avoid doing this in memory due to exponential size ») — exact lui aussi (0.8301 sur la même requête), à condition d'appeler d'abord `setTempDirectory(...)`, sans quoi `query` lève une `NullPointerException` ; et `IteratingMlnReasoner`, qui appelle un autre reasoner `N` fois et en moyenne les résultats.

In [7]:
// --- Cell[19] : comparaison exact (enumeration) vs approximatif (sampling) ---
using org.tweetyproject.logics.commons.syntax;
using org.tweetyproject.logics.commons.syntax.interfaces;
using org.tweetyproject.logics.fol.syntax;
using org.tweetyproject.logics.mln.syntax;
using org.tweetyproject.logics.mln.reasoner;
using System.Globalization;

string Inv(FormattableString fs) => fs.ToString(CultureInfo.InvariantCulture);

// Reconstitution du reseau social 3 personnes (meme modele que la cell[8])
var sig = new FolSignature();
var Smokes = new Predicate("Smokes", 1);
var Cancer = new Predicate("Cancer", 1);
var Friends = new Predicate("Friends", 2);
sig.add(Smokes); sig.add(Cancer); sig.add(Friends);
var anna = new Constant("anna"); var bob = new Constant("bob"); var carl = new Constant("carl");
sig.add(anna); sig.add(bob); sig.add(carl);
var X = new Variable("X"); var Y = new Variable("Y");

FolFormula At(Predicate p, params Term[] t) => (FolFormula)new FolAtom(p, t);

var mln = new MarkovLogicNetwork();
mln.add(new MlnFormula(
    (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(At(Smokes, X), At(Cancer, X)),
    java.lang.Double.valueOf(3.0)));
mln.add(new MlnFormula(
    (FolFormula)new org.tweetyproject.logics.fol.syntax.Implication(
        new org.tweetyproject.logics.fol.syntax.Conjunction(At(Friends, X, Y), At(Smokes, X)),
        At(Smokes, Y)),
    java.lang.Double.valueOf(2.0)));
mln.add(new MlnFormula(At(Smokes, anna)));
mln.add(new MlnFormula(At(Friends, anna, bob)));
mln.add(new MlnFormula(At(Friends, bob, carl)));

var atome = At(Cancer, carl);
Console.WriteLine("Atome requete : P(Cancer(carl))");
Console.WriteLine();

// (a) Reasoner exact (enumeration naive) - deterministe
var sw1 = System.Diagnostics.Stopwatch.StartNew();
var rn = new ApproximateNaiveMlnReasoner(200000L, 200000L);
double pExact = rn.query(mln, atome).doubleValue();
sw1.Stop();

// (b) Reasoner par echantillonnage (precision=0.01, 1000 tests positifs)
var sw2 = System.Diagnostics.Stopwatch.StartNew();
var rs = new SimpleSamplingMlnReasoner(0.01, 1000);
double pSamp = rs.query(mln, atome).doubleValue();
sw2.Stop();

Console.WriteLine(Inv($"  ApproximateNaive (exact) : {pExact:F4}   en {sw1.Elapsed.TotalSeconds:F2}s"));
Console.WriteLine(Inv($"  SimpleSampling   (approx): {pSamp:F4}   en {sw2.Elapsed.TotalSeconds:F2}s"));
Console.WriteLine(Inv($"  Ecart absolu              : {Math.Abs(pExact - pSamp):F4}"));
Console.WriteLine();
Console.WriteLine("=> Le sampling estime la meme marginale par tirage : sa valeur fluctue d'un run a l'autre.");
Console.WriteLine("   Son cout est domine par le nombre de tirages que fixe sa regle d'arret,");
Console.WriteLine("   pas par le nombre de mondes. Pour un domaine petit, l'exact reste preferable : deterministe.");


Atome requete : P(Cancer(carl))


  ApproximateNaive (exact) : 0.8301   en 5.93s


  SimpleSampling   (approx): 0.8438   en 0.43s


  Ecart absolu              : 0.0137


=> Le sampling estime la meme marginale par tirage : sa valeur fluctue d'un run a l'autre.


   Son cout est domine par le nombre de tirages que fixe sa regle d'arret,


   pas par le nombre de mondes. Pour un domaine petit, l'exact reste preferable : deterministe.


### Interprétation : exact vs approximatif

**Sortie réelle** : la cellule précédente, mesurée au Stopwatch sur un domaine d'environ 15 atomes ground. Les durées absolues dépendent de la machine ; la valeur exacte, elle, n'en dépend pas.

**Lecture** :

- Le reasoner **exact** (`ApproximateNaiveMlnReasoner`) énumère les interprétations de Herbrand et calcule la marginale rigoureuse — ici `P(Cancer(carl)) = 0.8301`, la même valeur que la cell[8] (même modèle). Coût exponentiel en théorie (`2^n` mondes), mais praticable sur un petit domaine. Le chronométrage compare les deux approches **sur ce run** : les deux durées se lisent dans la sortie et varient avec la machine, la charge et l'ordre d'exécution (premier appel compilé à chaud ou non) — ne retenez que l'ordre de grandeur.
- Le reasoner **par sampling** (`SimpleSamplingMlnReasoner(0.01, 1000)`) estime la marginale par tirage Monte-Carlo : sa valeur **fluctue d'une exécution à l'autre** (variance d'échantillonnage, non seedé), et son critère d'arrêt — variation sous la précision pendant 1000 tirages consécutifs — ne garantit pas la convergence. Sur ce run, il répond en moins d'une seconde là où l'énumération en prend plusieurs : c'est le compromis du sampling, vitesse contre exactitude, et l'écart mesuré avec la valeur exacte reste de l'ordre du centième. L'écart à la vraie valeur se lit dans la sortie, pas dans cette prose : le jumeau Python tire sa propre estimation sur la même requête dans sa cellule de comparaison, différente à chaque ré-exécution comme ici — autant de tirages légitimes du même estimateur autour de la vraie valeur 0.8301, et non deux mesures à rapprocher chiffre à chiffre.
- **Une limite plus grave, mesurée dans la cellule de mesure (bloc D)** : sur un réseau dont un monde pèse plus de `Long.MAX_VALUE` (ici `e^48 ≈ 7×10^20` à quatre personnes), sa source — qui accumule la masse satisfaite dans un `long` — sature, et l'estimation s'effondre jusqu'à deux ordres de grandeur sous la référence. Le témoin est un fait strict, vrai dans tout monde valide : il devrait sortir à 1 et sort à quelques centièmes. Ce n'est plus de la variance, c'est un débordement arithmétique ; sur un réseau à poids modérés, l'estimateur fluctue simplement.
- **Quand utiliser quoi** : énumération pour le debug, la validation et les petits graphes ; sampling pour les grands domaines où l'énumération explose — en acceptant la variance, et en contrôlant la convergence sur un cas dont on connaît la référence.

> Comme au notebook 2 (Sat4j portable vs CaDiCaL natif), le choix du reasoner est une **décision d'ingénierie** pilotée par la taille du domaine : exactitude quand elle est atteignable, approximation contrôlée quand elle ne l'est pas.

## Exercices (#2161 — 3 exos conformes)

Les exercices suivent la convention `.claude/rules/three-exercises-per-notebook.md` : ils sont **stubbés** (sans `raise NotImplementedError`, cf règle C.1) et **répartis** dans le notebook, chacun précédé d'un contexte et d'objectifs.

### Exercice 1 : Étendre le réseau social

**Contexte** : Le réseau de la partie 2 contient 3 personnes (anna, bob, carl). On veut y ajouter un 4ᵉ individu **dave** et observer la propagation de l'influence.

**Objectif** :

- Ajouter `dave` à la signature
- Ajouter le fait strict `Friends(carl, dave)`
- Requérir `P(Smokes(dave))` et `P(Cancer(dave))` après la propagation
- Comparer `P(Smokes(bob))`, `P(Smokes(carl))` et `P(Smokes(dave))` à celles du modèle à 3 personnes : l'égalité bob = carl observée en cell[8] survit-elle à l'ajout d'un individu ?

**Indice 1 — le piège du domaine** : avec 4 personnes, le domaine porte 2 × 4 + 16 = 24 atomes ground, soit 2^24 ≈ 16,7 millions de mondes — au-delà de la capacité (200000, 200000) utilisée jusqu'ici. `ApproximateNaiveMlnReasoner` bascule alors en régime aléatoire, **sans avertir**, et sa normalisation y est défectueuse (mécanisme en cell[11], bloc B) : le fait strict `P(Smokes(anna))` sort en dessous de 1, et les marginales s'écartent de la référence d'un appel à l'autre. Le bloc D de cell[10] mesure exactement ce domaine : la masse totale y reste proche de 1 (peu de doublons, car les tirages valides sont rares devant les mondes valides), et l'écart à la référence y est de l'ordre du centième. Ne lisez donc pas ces valeurs comme une référence : contrôlez d'abord le fait strict, puis comparez à l'énumération exhaustive de l'indice 2. `SimpleSamplingMlnReasoner(0.01, 1000)` n'est pas une alternative ici : sa source accumule la masse satisfaite dans un `long`, or le poids maximal d'un monde de ce réseau (`e^48`) dépasse `Long.MAX_VALUE` — le numérateur sature, et la dernière colonne du bloc D de cell[10] le montre, le fait strict compris. **Autre piège** : `query(mln, formule)` utilise la **signature minimale** du MLN — si `dave` n'apparaît dans aucune formule, il est ignoré. Passez la signature explicitement : `query(mln, formule, sig)`.

**Indice 2 — la référence** : une énumération exhaustive du joint, indépendante du moteur, donne `P(Smokes) = 1.0000 / 0.7169 / 0.6779 / 0.7169` pour anna / bob / carl / dave, et `P(Cancer) = 0.9526 / 0.8245 / 0.8068 / 0.8245` — le bloc D de cell[10] imprime ce tableau. bob et carl se séparent, bob et dave s'égalisent : l'égalité de la cell[8] n'était pas une symétrie structurelle. Ne préjugez pas du signe de l'écart : c'est la mesure qui répond.

### Exercice 2 : Un MLN de diagnostic médical

**Contexte** : On veut modéliser un raisonnement de diagnostic : on observe un symptôme, et plusieurs maladies peuvent l'expliquer avec des tendances différentes.

**Objectif** :

- Créer un MLN sur un domaine d'une seule personne `marie` avec :
  - `Fievre(X) => Grippe(X)` (poids 2.0), `Fievre(X) => Covid(X)` (poids 2.5) : de la fièvre **vers** la maladie — la direction du diagnostic ;
  - `Fievre(X) => TestPositif(X)` (poids 1.5)
- Ajouter le fait strict : `Fievre(marie)` (observé)
- Requérir `P(Grippe(marie))` et `P(Covid(marie))` — quelle maladie est la plus probable ?
- Ré-exécuter en échangeant les deux poids : la conclusion s'inverse-t-elle ?

**Indice** : un seul individu dans le domaine → les règles ne se groundent qu'une fois ; sur `Grippe`, seule la règle de poids 2.0 est en jeu, donc `P(Grippe(marie)) = sigma(2.0)`.

**Contrôle — le piège de la direction** : construisez ensuite le MLN avec les implications dans l'autre sens, `Grippe(X) => Fievre(X)` [2.0] et `Covid(X) => Fievre(X)` [2.5], en gardant `Fievre(marie)` observé, et requérez les deux mêmes marginales. Une règle `maladie => symptome` dont le symptôme est **déjà observé** est satisfaite dans tous les mondes : les poids n'ont alors **aucun** effet. C'est la différence entre expliquer un symptôme et l'utiliser pour diagnostiquer.

### Exercice 3 : Approcher le modèle à règle stricte

**Contexte** : Dans la partie 3, la marginale `P(Cancer(bob))` augmente avec le poids de `Smokes(X) => Cancer(X)`. Une règle stricte impose l'implication, mais pas nécessairement `Cancer(bob)` : bob ne fume pas avec certitude.

**Objectif** :

- Calculez `P(Cancer(bob))` avec la règle stricte (constructeur `MlnFormula` sans poids), en conservant les autres règles et faits
- Calculez la même marginale pour les poids `[3, 5, 7, 10, 15, 20]`
- Identifiez le premier poids testé dont la marginale diffère de celle du modèle strict de moins de `0.01`
- Expliquez pourquoi cette convergence n'impose pas une marginale égale à 1

**Indice** : reprenez la construction du sweep de la cell[13] et comparez les marginales par leur écart absolu au modèle strict — le seuil dépend de la tolérance choisie, non d'une valeur cible fixée à l'avance.

In [8]:
// --- Exercice 1 : étendre le réseau social à un 4ᵉ individu ---
// TODO étudiant : ajoutez 'dave' à la signature (FolSignature simple, sans sort Person), ajoutez
// le fait strict Friends(carl, dave), puis requérez P(Smokes(dave)) et P(Cancer(dave)).
// Attention : 24 atomes ground = 2^24 mondes, au-dela de la capacite (200000, 200000) de la
// cell[8] : ApproximateNaiveMlnReasoner bascule en regime aleatoire, dont la normalisation est
// defectueuse (cf. cell[11], bloc B) -- un fait strict peut sortir < 1. Controle : P(Smokes(anna)) doit
// rester a 1 ; les marginales se comparent a l'enumeration exhaustive de cell[10] (bloc D) et
// cell[21] (Indice 2), pas a SimpleSamplingMlnReasoner. Passez la signature explicitement :
// query(mln, formule, sig) (la surcharge a 2 arguments utilise la signature MINIMALE, qui
// ignore dave s'il n'apparait dans aucune formule).

// Cellule stub C.1-conforme : pas de raise NotImplementedError (le notebook
// doit s'executer end-to-end). L'etudiant remplace ce commentaire par
// son implementation en utilisant les APIs TweetyMLN exposees en cell[8] (marginales) et cell[13] (sweep de poids).
//
// using org.tweetyproject.logics.fol.syntax;
// using org.tweetyproject.logics.mln.syntax;
// using org.tweetyproject.logics.mln.reasoner;
// ... votre code ici ...

Console.WriteLine("// --- Exercice 1 : étendre le réseau social à un 4ᵉ individu --- = Exercice a completer");

// --- Exercice 1 : étendre le réseau social à un 4ᵉ individu --- = Exercice a completer


In [9]:
// --- Exercice 2 : MLN de diagnostic médical ---
// TODO étudiant : construisez un MLN de diagnostic médical (domaine : une seule constante 'marie'
// + prédicats Grippe, Covid, Fievre, TestPositif). Regles ponderees, dans la direction du
// DIAGNOSTIC : Fievre(X) => Grippe(X) [w=2.0], Fievre(X) => Covid(X) [w=2.5],
// Fievre(X) => TestPositif(X) [w=1.5]. Fait strict : Fievre(marie).
// Requerez P(Grippe(marie)) et P(Covid(marie)), puis echangez les deux poids et re-executez.
// Controle : reconstruisez le meme MLN avec Grippe(X) => Fievre(X) / Covid(X) => Fievre(X)
// (maladie => symptome), Fievre(marie) toujours observe : que deviennent les marginales,
// et que devient l'effet des poids ?

// Cellule stub C.1-conforme : pas de raise NotImplementedError (le notebook
// doit s'executer end-to-end). L'etudiant remplace ce commentaire par
// son implementation en utilisant les APIs TweetyMLN exposees en cell[8] (marginales) et cell[13] (sweep de poids).
//
// using org.tweetyproject.logics.fol.syntax;
// using org.tweetyproject.logics.mln.syntax;
// using org.tweetyproject.logics.mln.reasoner;
// ... votre code ici ...

Console.WriteLine("// --- Exercice 2 : MLN de diagnostic médical --- = Exercice a completer");

// --- Exercice 2 : MLN de diagnostic médical --- = Exercice a completer


In [10]:
// --- Exercice 3 : approcher le modèle à règle stricte ---
// TODO étudiant : calculez P(Cancer(bob)) avec la regle stricte (MlnFormula sans poids),
// puis pour les poids [3, 5, 7, 10, 15, 20] ; identifiez le premier poids dont la marginale
// differe de celle du modele strict de moins de 0.01, et expliquez pourquoi la convergence
// n'implique pas une marginale egale a 1.

// Cellule stub C.1-conforme : pas de raise NotImplementedError (le notebook
// doit s'executer end-to-end). L'etudiant remplace ce commentaire par
// son implementation en utilisant les APIs TweetyMLN exposees en cell[8] (marginales) et cell[13] (sweep de poids).
//
// using org.tweetyproject.logics.fol.syntax;
// using org.tweetyproject.logics.mln.syntax;
// using org.tweetyproject.logics.mln.reasoner;
// ... votre code ici ...

Console.WriteLine("// --- Exercice 3 : approcher le modèle à règle stricte --- = Exercice a completer");

// --- Exercice 3 : approcher le modèle à règle stricte --- = Exercice a completer


## Résumé

Ce notebook a couvert :

- **Le concept de MLN** : une formule FOL + un poids, unifiant logique symbolique et raisonnement statistique (Richardson & Domingos 2006). Poids = `+∞` (strict, FOL classique) ou `w` fini (tendance statistique, coût `exp(-w)` par violation).
- **La construction d'un MLN** : `MarkovLogicNetwork` (extends `BeliefSet<MlnFormula, FolSignature>`) qui contient des `MlnFormula` (stricte ou pondérée) construites à partir de `FolFormula`.
- **L'interrogation** : `ApproximateNaiveMlnReasoner.query(mln, formula)` retourne une probabilité marginale — **exacte** tant que `2^N` tient dans la capacité du constructeur, sous-évaluée en régime aléatoire. Alternatives : `SimpleSamplingMlnReasoner` (Monte-Carlo), `AlchemyMlnReasoner` (wrapper natif, optionnel).
- **Le spectre logique ↔ statistique** : `w = 0` = aucune contrainte ; `w` croissant = tendance de plus en plus forte ; `w = +∞` = implication imposée, sans que la marginale de la conclusion atteigne nécessairement 1 (0.8573 sur ce réseau).
- **L'exception gérée nativement** : une règle stricte (poids `+∞`) domine une règle pondérée concurrente dans le même domaine (cf paradoxe du pingouin).

**Cas d'usage industriels** :

- **Diagnostic médical** : combiner des symptômes observés (stricts) avec des connaissances médicales générales (pondérées)
- **Extraction d'information** : fusionner plusieurs extractions bruitées avec des poids de confiance
- **Détection de fraude** : règles métier (strictes) + signaux statistiques (pondérés)
- **Réseaux sociaux** : modéliser la diffusion d'influence ou de comportements

**Limites des MLN** :

- L'inférence est **#P-complète** dans le cas général (Domingos & Richardson 2006)
- Le choix des poids est **empirique** (appris par optimisation ou fixé à la main)
- Pour des domaines > 30 atomes, l'énumération naïve est intractable → préférer MCMC/sampling

## Pour aller plus loin

- **Notebook suivant** : [Tweety-11-Causal-Python.ipynb](Tweety-11-Causal-Python.ipynb) — réseaux causaux (Pearl, do-calculus)
- **Référence** : Richardson, M., & Domingos, P. (2006). *Markov logic networks*. Machine Learning, 62(1-2), 107-136.
- **Documentation Tweety** : <https://tweetyproject.org/api/1.30/org/tweetyproject/logics/mln/package-summary.html>
- **Code source** : `dotnet-build/build-tweety-mln-shade.pom.xml` (Maven shade) + `build-TweetyMlnShade.csproj` (IKVM) → `org.tweetyproject.tweety-mln.dll` (≈14 MB)